# Post-Processing: A Way to Recalculate Scores

For example: Null Handling post-processor that reclassifies null/empty values.

## The Problem: Constrained-Output Tools

Previous examples assumed the extractor can omit fields (key and value), or hallucinate extra fields (key and value).
With this assumption, the default scoring system defines:
- **Omission** = field present in gold but missing from extracted (key absent in extracted). Penalizes recall.
- **Hallucination** = field missing from gold but present in extracted (key absent in gold). Penalizes precision.
- **Mismatch** = field present in both, but values differ. Penalizes both precision and recall.

With constrained-output tools, every schema field is always present in extracted -- so
**omission** (missing key) never occurs. Hallucination can still occur for optional
fields (gold omits the key, extracted has it), but the more common issue is that
null/empty values are treated as regular values:

- gold=`"PVD"`, extracted=`null` -> **mismatch** (but the extractor didn't guess wrong -- it gave up)
- gold=`null`, extracted=`"PVD"` -> **mismatch** (but gold has no answer -- the extractor invented one)
- gold=`null`, extracted=`null` -> **match** (but there's nothing meaningful to match on)

There's no distinction between "wrong answer" and "no answer." Precision equals
recall equals accuracy. Not useful.

## The Solution: `reclassify_nulls`

A **post-processor** that runs after scoring and reclassifies results based on
whether gold/extracted values are "absent" (null, empty string, etc.):

| Gold | Extracted | Default scoring | With null handling scoring                                  |
|------|-----------|-----------------|-------------------------------------------------------------|
| real value | real value | match/mismatch  | unchanged                                                   |
| real value | absent | mismatch        | **omission** (extractor gave up)                            |
| absent | real value | mismatch        | **hallucination** (extractor invented)                      |
| absent | absent | match           | **skipped or match depends on config** (nothing to measure) |

Now precision != recall. The metrics tell you: "the extractor is perfect when it
answers (precision), but only answers half the time (recall)."

In short, the default scoring system treats omission and hallucination as missing or extra **keys**. The null handling post-processor extends this to treat certain **values** (null, empty string, etc.) as absent too -- so "key present but value is null" is treated the same as "key missing."

## Data

Simulates constrained output: every field is always present. The LLM uses `null`
or `""` to mean "I couldn't find this."

In [1]:
GOLD = [
    # Record 0: gold has all fields
    {"method": "PVD", "temp": 300, "notes": "Substrate pre-heated."},
    # Record 1: gold has no notes (null)
    {"method": "CVD", "temp": 500, "notes": None},
]

EXTRACTED = [
    # Extractor gave up on temp (null) and notes ("")
    {"method": "PVD", "temp": None, "notes": ""},
    # Extractor invented notes that don't exist in gold
    {"method": "CVD", "temp": 500, "notes": "N/A"},
]

## Step 1: Default Scoring (no post-processing)

`null` and `""` are treated as regular values.

In [2]:
from struct_extract_eval import evaluate, infer_schema, annotate_xeval


def show_results(run, title=""):
    """Print run summary and per-record field results."""
    if title:
        print(title)
    print(f"  Mean F1: {run.mean_f1:.3f}  P: {run.mean_precision:.3f}  R: {run.mean_recall:.3f}")
    print()
    for record in run.records:
        print(f"  Record {record.record_id} (F1: {record.f1:.3f}):")
        for fr in record.field_results:
            g = str(fr.gold_value)
            e = str(fr.extracted_value)
            reason = f"  ({fr.reason})" if fr.reason else ""
            print(f"    {fr.path:<10} gold={g:<20} ext={e:<20} {fr.status:<15}{reason}")


schema = infer_schema(GOLD)
annotate_xeval(schema)

run_default = evaluate(GOLD, EXTRACTED, schema=schema)
show_results(run_default, "Default scoring (null is a value):")

Default scoring (null is a value):
  Mean F1: 0.500  P: 0.500  R: 0.500

  Record 0 (F1: 0.333):
    method     gold=PVD                  ext=PVD                  match          
    notes      gold=Substrate pre-heated. ext=                     mismatch         (mismatch)
    temp       gold=300                  ext=None                 mismatch         (type_error)
  Record 1 (F1: 0.667):
    method     gold=CVD                  ext=CVD                  match          
    notes      gold=None                 ext=N/A                  mismatch         (mismatch)
    temp       gold=500                  ext=500                  match          


Everything is a match or mismatch. No omissions, no hallucinations. Precision == Recall.

## Step 2: With Null Handling

Configure which values mean "absent" and pass `reclassify_nulls` as a post-processor.

In [3]:
from struct_extract_eval.postprocess import NullHandling, reclassify_nulls

config = NullHandling(
    absent_values=[None, ""],  # both null and empty string mean "no answer"
    both_absent_skip=True,     # null vs null = skip (excluded from metrics), otherwise, null vs null = match (score 1.0)
)

run_null = evaluate(
    GOLD, EXTRACTED, schema=schema,
    post_process=lambda frs: reclassify_nulls(frs, config),
)

show_results(run_null, "With null handling (null/empty = absent):")

With null handling (null/empty = absent):
  Mean F1: 0.650  P: 0.833  R: 0.667

  Record 0 (F1: 0.500):
    method     gold=PVD                  ext=PVD                  match          
    notes      gold=Substrate pre-heated. ext=                     omission         (extracted is absent)
    temp       gold=300                  ext=None                 omission         (extracted is absent)
  Record 1 (F1: 0.800):
    method     gold=CVD                  ext=CVD                  match          
    notes      gold=None                 ext=N/A                  hallucination    (gold is absent)
    temp       gold=500                  ext=500                  match          


Result:
- **Record 0, `temp`:** gold=300, extracted=null -> **omission** (extractor gave up). Hurts recall.
- **Record 0, `notes`:** gold="Substrate pre-heated.", extracted="" -> **omission**. Hurts recall.
- **Record 1, `notes`:** gold=null, extracted="N/A" -> **hallucination** (extractor invented). Hurts precision. If your extractor uses "N/A" to mean "no answer" (same as null), add it to `absent_values`: `NullHandling(absent_values=[None, "", "N/A"])`.


## Configuration Options

### `absent_values`

Which values count as "absent." All values in the list are treated as equivalent:

```python
NullHandling(absent_values=[None])          # only null
NullHandling(absent_values=[None, ""])      # null or empty string
NullHandling(absent_values=[None, "", []])  # null, empty string, or empty list
```

This means `None` vs `""` = both absent (same category), not a mismatch.

### `both_absent_skip`

What happens when both gold and extracted are absent:

```python
both_absent_skip=True   # skip -- excluded from metrics entirely (default)
both_absent_skip=False  # match -- counts as a correct extraction (score 1.0)
```

## Writing Your Own Post-Processor

`post_process` accepts any function `(list[FieldResult]) -> list[FieldResult]`.
You can chain multiple post-processors:

```python
from struct_extract_eval.postprocess import reclassify_nulls, propagate_batch_errors

def my_post_process(frs):
    # If the LLM judge (batch comparator) failed on any field in a record,
    # its other "successful" results in the same batch may be unreliable
    # (e.g. the judge miscounted or misaligned responses). This marks ALL
    # fields from that tainted batch as batch_error so they don't pollute
    # metrics with potentially wrong scores.
    propagate_batch_errors(frs)
    reclassify_nulls(frs, config)
    return frs

result = evaluate(gold, extracted, schema, post_process=my_post_process)
```

Post-processors run **after** batch comparator dispatch. This package includes two
built-in post-processors: `reclassify_nulls` (null handling) and
`propagate_batch_errors` (batch error handling). You can also write your own.